# Pre trained

In [1]:
#from transformers import DistilBertTokenizer
#
#tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased', do_lower_case=True)
#cls_token_id = tokenizer.cls_token_id  # or tokenizer.convert_tokens_to_ids("[CLS]")
#mask_token_id = tokenizer.mask_token_id  # or tokenizer.convert_tokens_to_ids("[MASK]")
#pad_token_id = tokenizer.pad_token_id  # or tokenizer.convert_tokens_to_ids("[MASK]")
#vocab_size=tokenizer.vocab_size

In [2]:
#tokenizer.encode("you are bruh", add_special_tokens=True)

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import numpy as np

In [4]:
#train_set_large = train_set.sample(frac=1).reset_index(drop=True)
large_set = pd.read_csv("995,000_rows.csv").sample(frac=0.001).reset_index(drop=True)
large_set = large_set[["content", "type", "title"]].dropna()

/var/folders/89/bk15wb4x33g0_6ndpf2c8rqh0000gn/T/ipykernel_49294/1625276648.py:2: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  large_set = pd.read_csv("995,000_rows.csv").sample(frac=0.001).reset_index(drop=True)


In [5]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import NFD, StripAccents, Lowercase

tokenizer = Tokenizer(BPE())
tokenizer.normalizer = NFD()
tokenizer.pre_tokenizer = Whitespace()

vocab_size=40000
trainer = BpeTrainer(vocab_size=vocab_size, special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"])

tokenizer.train_from_iterator(large_set["content"], trainer=trainer)

tokenizer.save("bpe_tokenizer.json")

cls_token_id = tokenizer.token_to_id("[CLS]")
mask_token_id = tokenizer.token_to_id("[MASK]")
pad_token_id = tokenizer.token_to_id("[PAD]")

In [6]:
label_map = {
    "fake": "fake",
    "satire": None,
    "bias": None,
    "conspiracy": "fake",
    "state": None,
    "junksci": "fake",
    "hate": None,
    "clickbait": None,
    "unreliable": None,
    "political": "reliable",
    "reliable": "reliable",
    "unknown": None,
}

In [7]:
large_set["new_labels"] = [label_map.get(n, None) for n in large_set["type"]]
#large_set["content_tokens"] = fn.tokenize(large_set["content"])

In [8]:
import random

def mask_random_elements(sequence, mask_probability=0.15):
    masked_sequence = sequence[:]
    for idx in range(len(sequence)):
        if random.random() < mask_probability and sequence[idx] != pad_token_id:
            masked_sequence[idx] = mask_token_id
    return masked_sequence

max_seq_len = 300

def tokenize(text):
    tokens = tokenizer.encode(text)
    tokens = [cls_token_id] + tokens.ids
    tokens = tokens[:max_seq_len]
    while len(tokens) < max_seq_len:
        tokens.append(pad_token_id)
    return tokens

text_ids = [tokenize(text) for text in large_set["content"]]
text_ids_masked = [mask_random_elements(text_id) for text_id in text_ids]
att_masks = [[id > 0 for id in ids] for ids in text_ids]

In [9]:
from sklearn.model_selection import train_test_split

labels = [1.0 if n == "fake" else 0 for n in large_set["new_labels"]]

train_x, test_val_x, train_y, test_val_y = train_test_split(text_ids, labels, test_size=0.2)
train_m, test_val_m = train_test_split(att_masks, test_size=0.2)
train_mlm, test_val_mlm = train_test_split(text_ids_masked, test_size=0.2)

test_x, val_x, test_y, val_y = train_test_split(test_val_x, test_val_y, test_size=0.5)
test_m, val_m = train_test_split(test_val_m, test_size=0.5)

In [10]:
train_x = torch.tensor(train_x)
train_mlm = torch.tensor(train_mlm)
test_x = torch.tensor(test_x)
val_x = torch.tensor(val_x)

train_y = torch.tensor(train_y)
test_y = torch.tensor(test_y)
val_y = torch.tensor(val_y)

train_m = torch.tensor(train_m)
test_m = torch.tensor(test_m)
val_m = torch.tensor(val_m)

In [11]:
#train_x = torch.tensor([n for n in train_set_large["tokens"]])

In [12]:
#train_y = torch.tensor([int(n == "fake") for n in train_set_large["new_labels"]])

In [13]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

batch_size = 32

train_data = TensorDataset(train_x, train_m, train_y, train_mlm)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

val_data = TensorDataset(val_x, val_m, val_y)
val_sampler = SequentialSampler(val_data)
val_dataloader = DataLoader(val_data, sampler=val_sampler, batch_size=batch_size)

In [16]:
from transformers import DistilBertModel, DistilBertConfig
from torch.optim import Adam, AdamW
from torch import nn, functional as F
import math

num_labels = 2
device = torch.device("mps")

class BertClassifier(nn.Module):
    def __init__(self):
        super(BertClassifier, self).__init__()
        #self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased', output_attentions=False, output_hidden_states=False)
        #self.out = nn.Linear(self.bert.config.dim, 1)
        
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8)
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=6)

        self.embeddings = nn.Embedding(num_embeddings=vocab_size, embedding_dim=512, padding_idx=0)
        self.positional_encoding = torch.zeros([max_seq_len, 512])#, requires_grad=True)
        position = torch.arange(0, max_seq_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, 512, 2).float() * -(math.log(10000.0) / 512))
        self.positional_encoding[:, 0::2] = torch.sin(position * div_term)
        self.positional_encoding[:, 1::2] = torch.cos(position * div_term)

        self.preclassifier = nn.Linear(512, 512)
        self.classifier = nn.Linear(512, 1)
        self.distribution = nn.Linear(512, vocab_size)

    def hidden(self, seq, **kwargs):
        x = self.embeddings(seq)
        expanded_positional_encoding = self.positional_encoding.unsqueeze(0)  # Shape: (1, sequence_length, embedding_size)
        expanded_positional_encoding = expanded_positional_encoding.expand(x.shape[0], -1, -1).to(device)  # Shape: (batch_size, sequence_length, embedding_size)
        x += expanded_positional_encoding
        return self.transformer_encoder(x, **kwargs)
    
    def classify(self, x, **kwargs):
        #out = self.bert(x, **kwargs)
        #return self.out(out.last_hidden_state[:,0]).tanh().flatten()
        out = self.hidden(x, **kwargs)
        return self.classifier(self.preclassifier(out[:,0]).relu())

    def mlm(self, x, **kwargs):
        out = self.hidden(x)
        return self.distribution(out)
    
model = BertClassifier()

model = model.to(device)
model.train()

BertClassifier(
  (encoder_layer): TransformerEncoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
    )
    (linear1): Linear(in_features=512, out_features=2048, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (linear2): Linear(in_features=2048, out_features=512, bias=True)
    (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (dropout1): Dropout(p=0.1, inplace=False)
    (dropout2): Dropout(p=0.1, inplace=False)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
  

In [17]:
learning_rate = 1e-4
adam_epsilon = 1e-8

optimizer = AdamW(model.named_parameters(), lr=learning_rate, eps=adam_epsilon)

In [18]:
from transformers import get_linear_schedule_with_warmup

num_epochs = 4
total_steps = len(train_dataloader) * num_epochs

scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [19]:
import time

def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs


In [20]:
def train(finetune=False):
    train_losses = []
    val_losses = []
    num_mb_train = len(train_dataloader)
    num_mb_val = len(val_dataloader)
    missed_batches = 0
    
    if num_mb_val == 0:
        num_mb_val = 1
    
    for n in range(num_epochs):
        train_loss = 0
        val_loss = 0
        start_time = time.time()
        
        for k, (mb_x, mb_m, mb_y, mb_mlm) in enumerate(train_dataloader):
            optimizer.zero_grad()
            model.train()
    
            if mb_x.shape[0] != 32:
                missed_batches += 1
                continue
            
            mb_x = mb_x.to(device)
            mb_m = mb_m.to(device)
            mb_y = mb_y.to(device)
            mb_mlm = mb_mlm.to(device)
    
            if finetune:
                outputs = model.classify(mb_x, src_key_padding_mask=mb_m.transpose(0,1))
                loss = torch.nn.functional.mse_loss(outputs.view(-1), mb_y)
                loss.backward()
                #torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            else:
                outputs = model.mlm(mb_mlm, attention_mask=mb_m)
                #print(outputs.shape, mb_x.shape)
                loss = torch.nn.functional.cross_entropy(outputs.view(-1, vocab_size), mb_x.view(-1))
                loss.backward()
            
            optimizer.step()
            scheduler.step()
            
            train_loss = loss.data
    
            # Track time for every batch
            elapsed_time = time.time() - start_time
            batches_done = k + 1
            batches_total = len(train_dataloader)
            
            # Estimate time remaining (ETA)
            remaining_batches = batches_total - batches_done
            eta_seconds = (elapsed_time / batches_done) * remaining_batches
            eta_str = str(time.strftime("%H:%M:%S", time.gmtime(eta_seconds)))
    
            train_losses.append(train_loss.cpu())
    
            # Print current status and ETA
            print(f"Loss: {float(np.mean(train_losses)):.2f} - ETA of epoch: {eta_str}", end='\r')
        
        print (f"\n missed {missed_batches} batches")
        
        with torch.no_grad():
            model.eval()
            
            for k, (mb_x, mb_m, mb_y) in enumerate(val_dataloader):
                mb_x = mb_x.to(device)
                mb_m = mb_m.to(device)
                mb_y = mb_y.to(device)
    
                if mb_x.shape[0] != 32:
                    missed_batches += 1
                    continue
                outputs = model.classify(mb_x, src_key_padding_mask=mb_m.transpose(0,1))
                loss = torch.nn.functional.mse_loss(outputs.view(-1), mb_y)
                
                val_loss += loss.data / num_mb_val
                
            print ("Validation loss after itaration %i: %f" % (n+1, val_loss))
            val_losses.append(val_loss.cpu())
        
        end_time = time.time()
        epoch_mins, epoch_secs = epoch_time(start_time, end_time)
        print(f'Time: {epoch_mins}m {epoch_secs}s')


train()
optimizer = Adam(model.named_parameters(), lr=learning_rate*0.1, eps=adam_epsilon)

train(finetune=True)

Loss: 8.35 - ETA of epoch: 00:00:016
 missed 1 batches
Validation loss after itaration 1: 0.198517
Time: 0m 33s
Loss: 7.84 - ETA of epoch: 00:00:01
 missed 3 batches
Validation loss after itaration 2: 0.198517
Time: 0m 27s
Loss: 7.53 - ETA of epoch: 00:00:01
 missed 5 batches
Validation loss after itaration 3: 0.198517
Time: 0m 31s
Loss: 7.33 - ETA of epoch: 00:00:01
 missed 7 batches
Validation loss after itaration 4: 0.198517
Time: 0m 25s
Loss: 0.18 - ETA of epoch: 00:00:00
 missed 1 batches
Validation loss after itaration 1: 0.198510
Time: 0m 12s
Loss: 0.18 - ETA of epoch: 00:00:00
 missed 3 batches
Validation loss after itaration 2: 0.198511
Time: 0m 12s
Loss: 0.18 - ETA of epoch: 00:00:00
 missed 5 batches
Validation loss after itaration 3: 0.198511
Time: 0m 12s
Loss: 0.18 - ETA of epoch: 00:00:00
 missed 7 batches
Validation loss after itaration 4: 0.198508
Time: 0m 12s


In [21]:
from matplotlib import pyplot as plt
%matplotlib inline

plt.figure()
plt.plot(train_losses)

NameError: name 'train_losses' is not defined

<Figure size 640x480 with 0 Axes>

In [ ]:
plt.figure()
plt.plot(val_losses)

In [23]:
batch_size = 32

test_data = TensorDataset(test_x, test_m)
test_sampler = SequentialSampler(test_data)
test_dataloader = DataLoader(test_data, sampler=test_sampler, batch_size=batch_size)

outputs = []
with torch.no_grad():
    model.eval()
    for k, (mb_x, mb_m) in enumerate(test_dataloader):
        mb_x = mb_x.to(device)
        mb_m = mb_m.to(device)
        output = model.classify(mb_x, src_key_padding_mask=mb_m.transpose(0,1))
        outputs.append(output)

outputs = torch.cat(outputs)

In [24]:
hidden_states = []
with torch.no_grad():
    model.eval()
    for k, (mb_x, mb_m) in enumerate(test_dataloader):
        mb_x = mb_x.to(device)
        mb_m = mb_m.to(device)
        output = model.hidden(mb_x, src_key_padding_mask=mb_m.transpose(0,1))[:,0]
        hidden_states.append(output)

hidden_states = torch.cat(hidden_states)
hidden_states

tensor([[nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        ...,
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan]], device='mps:0')

In [ ]:
predicted_values = torch.round(outputs)
predicted_values = predicted_values.cpu().view(-1).numpy()
true_values = test_y.numpy()

In [ ]:
import numpy as np

In [ ]:
test_accuracy = np.sum(predicted_values == true_values) / len(true_values)
print ("Test Accuracy:", test_accuracy)

In [ ]:
label_values = ["fake", "reliabel"]

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(true_values, predicted_values, target_names=[str(l) for l in label_values]))

In [ ]:
import itertools

# plot confusion matrix
# code borrowed from scikit-learn.org
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

cm_test = confusion_matrix(true_values, predicted_values)

np.set_printoptions(precision=2)

#plt.figure(figsize=(6,6))
#plot_confusion_matrix(cm_test, classes=label_values, title='Confusion Matrix - Test Dataset')
plt.figure(figsize=(6,6))
plot_confusion_matrix(cm_test, classes=label_values, title='Confusion Matrix - Test Dataset', normalize=True)